# 17 - The sky terms, and the pairs the model is asked to rank

**Purpose.** Run the analysis half of `protocols/06-sky-pair.md` on `data/session06` and publish
the three constants that cannot be measured with the cap on - `F_sky` per CFA plane, `t_dead`, and
as much of `eta_comb` as this repo can currently reach - together with the per-frame record of the
night and the four cells' predicted separations. It is the first session in this project that
points the camera at the sky.

**What it is not for.** `g(gain)` and `R(gain)` are **inputs**, consumed and never re-measured
here. They come from `results/cold_constants.json` and **not** from `ptc_constants.json`, because
this night was shot at a -20 C setpoint and session 02 published at -10 C: notebook `15` exists to
remove exactly that substitution. Not a gain sweep - two gains, chosen, not scanned. Not
`ceiling(gain)`. Not vignetting, focus, guiding or seeing, all of which MISSION scopes out. And
**not a picture**: no framing, filtering or processing decision below is made to improve one.

**Three things the night did not deliver, named here rather than discovered in section 8.**

1. **The setpoint was -20 C, not the protocol's -10 C.** Every frame, all night. That is why this
   notebook reads `cold_constants.json`, and why section 5 refuses to run without it.
2. **No bias brackets were shot.** The protocol asks for 20 at each gain at each end of the
   night, so that the pedestal comes from *this* night and the offset state can be assigned. There
   are none. The pedestal therefore comes from notebook `15`'s bench blocks at the same setpoint
   and gains - a real measurement, at the right temperature, on the wrong night. Section 5 states
   what that costs.
3. **The field rotated 179 degrees after the first few frames** - a meridian flip. Gate 4 of the
   protocol fixes the two ROIs for the whole night, and a flip moves them. Section 1 drops the
   pre-flip frames rather than defining a second pair of ROIs for a handful of them.

**And one the repo did not deliver.** `eta_comb` on registered lights, the SNR estimator's
repeatability, and the four ranked pairs all need frames **registered and integrated**. That is
PixInsight, `astropix/pixinsight.py` and `pjsr/` - build step 5, which does not exist yet. Section
7 publishes those three as nulls with reasons and says exactly what is needed to fill them. What
it does instead is compute each pair's predicted separation from **this session's own measured
constants** rather than from the inherited ones the protocol was written with, which is what tells
you which pairs are still tests once the real `t_dead` is known.

**Sections 1 to 4 and 6 need nothing but the frames.** Only section 5 needs notebook `15`.

## 1. The night, from the headers

Every frame's capture settings, read from the header and trusted - the protocol is what set them.
The *type* label is not trusted anywhere in this repo, but nothing here needs it: these are the
frames this project shot, to a protocol, and the cell grid is defined by gain and exposure.

Three things are settled here, in order: which frames the meridian flip cost, whether the cooler
held, and whether the four cells are still balanced after both.

In [ ]:
import datetime as dt
import json
import pathlib
import sys

import numpy as np
import pandas as pd
from astropy.io import fits as _afits

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import fits as F, spatial, stats

ROOT = pathlib.Path.cwd().parent
RESULTS = ROOT / "results"
DATA = ROOT / "data" / "session06"

FULL_SCALE = 4095                       # ADC counts; the units rule in CLAUDE.md
PLANES = list(spatial.PLANES)
SETPOINT_C = -20.0                      # what the night was actually shot at, not the protocol's
TEMP_BAND_C = 0.5                       # protocol: a frame outside this is flagged, not dropped
CELLS = [(50, 30.0), (50, 120.0), (200, 30.0), (200, 120.0)]
CELL_NAME = {(50, 30.0): "A", (50, 120.0): "B", (200, 30.0): "C", (200, 120.0): "D"}

# Gate 4: the two ROIs, fixed for the night and never moved.  The signal box is
# the one sessions 01, 02, 03 and 05 used, so a number from it is comparable
# with the bench.  The sky box is chosen once, from the data, in section 3.
SIGNAL_ROI = (1408, 568, 1024, 1024)    # x, y, w, h -- even throughout, or the Bayer phase shifts
SKY_BOX = 512

FRAMES_CSV = RESULTS / "sky_frames.csv"
PAIRS_CSV = RESULTS / "sky_pairs.csv"
CONSTANTS = RESULTS / "sky_constants.json"

files = sorted(DATA.glob("*.fit")) + sorted(DATA.glob("*.fits"))
assert files, f"no frames in {DATA}"

# Headers only in this pass: 160 full frames is 2.6 GB of reads and nothing here
# needs a pixel.  `fits.read` is the project's reader and is used from section 3
# on, where the pixels are actually wanted.
rows = []
for p in files:
    h = _afits.getheader(p)
    rows.append({
        "file": p.name, "gain": int(h["GAIN"]), "exptime": float(h["EXPTIME"]),
        "offset": int(h["OFFSET"]), "set_temp": float(h["SET-TEMP"]),
        "ccd_temp": float(h["CCD-TEMP"]), "date_obs": str(h["DATE-OBS"]),
        "rotator": float(h.get("ROTATOR", np.nan)), "focuspos": h.get("FOCUSPOS"),
        "object": str(h.get("OBJECT", "")).strip(),
    })

night = pd.DataFrame(rows).sort_values("date_obs").reset_index(drop=True)
night["t"] = pd.to_datetime(night.date_obs)
night["cell"] = [CELL_NAME.get((g, e)) for g, e in zip(night.gain, night.exptime)]
night["minutes"] = (night.t - night.t.iloc[0]).dt.total_seconds() / 60.0

print(f"{len(night)} frames, {night.t.iloc[0]} to {night.t.iloc[-1]} "
      f"({night.minutes.iloc[-1] / 60:.2f} h)")
print(f"object {sorted(night.object.unique())}, offset {sorted(night.offset.unique())}, "
      f"setpoint {sorted(night.set_temp.unique())}")
print()
print(night.groupby(["cell", "gain", "exptime"]).size().rename("frames").to_string())

### The meridian flip, and the frames it costs

The protocol is explicit: either avoid a flip inside the capture window, or redefine both ROIs
after it and treat the two halves as separate blocks. Deciding in advance is what stops the flip
becoming a silent confound.

Neither was done, so the decision is made here and made the cheap way. The pre-flip frames are
**dropped**, because defining a second pair of ROIs for a dozen frames buys less than it costs,
and because every cell still clears the protocol's floor of 24 afterwards.

The rule is stated before the count is looked at: **the night's modal camera angle is the night,
and every frame at a different angle is out.**

In [ ]:
MODAL_ANGLE = float(night.rotator.mode().iloc[0])
night["flipped"] = (night.rotator - MODAL_ANGLE).abs() > 1.0

print(f"camera angles present: {sorted(night.rotator.unique())}")
print(f"modal angle {MODAL_ANGLE:.0f} deg -- the night")
print(f"{int(night.flipped.sum())} frames at another angle, dropped")
print()
print(night[night.flipped].groupby("cell").size().rename("dropped").to_string())
print()
kept = night[~night.flipped].copy()
per_cell = kept.groupby("cell").size()
print("frames per cell after the drop:")
print(per_cell.to_string())

PROTOCOL_FLOOR = 24
assert per_cell.min() >= PROTOCOL_FLOOR, (
    f"cell {per_cell.idxmin()} has {per_cell.min()} frames, under the protocol's floor of "
    f"{PROTOCOL_FLOOR}.  That is a session that did not run -- reshoot rather than publishing "
    "a stack the eta_comb ladder cannot reach")
print(f"\nall four cells clear the floor of {PROTOCOL_FLOOR}.  The halves test wants an even "
      f"split: {per_cell.min() // 2} per half at worst")

### Gate 2 read back - did the cooler hold?

The protocol logs temperature per frame and flags a frame outside +/-0.5 C rather than silently
keeping it. On sky the ambient is not the bench's, and a TEC at its limit in September air is a
real possibility.

Note what this gate can and cannot say. It reads `CCD-TEMP`, which is the sensor reporting itself,
so it catches the cooler losing the band. It cannot catch the setpoint being wrong in the first
place - which is exactly what happened, and which no per-frame check would ever have found.

In [ ]:
kept["temp_off"] = (kept.ccd_temp - SETPOINT_C).abs()
kept["temp_flag"] = kept.temp_off > TEMP_BAND_C

print(f"setpoint {SETPOINT_C} C, band +/-{TEMP_BAND_C} C")
print(f"achieved: {kept.ccd_temp.min()} to {kept.ccd_temp.max()} C, "
      f"median {kept.ccd_temp.median()} C")
print(f"{int(kept.temp_flag.sum())} frames outside the band "
      f"({100 * kept.temp_flag.mean():.1f}%)")
print()
print(kept.groupby("cell").agg(min_c=("ccd_temp", "min"), max_c=("ccd_temp", "max"),
                               flagged=("temp_flag", "sum")).to_string())
print("\nthe setpoint itself was -20 C on every frame, against the protocol's -10 C.")
print("No per-frame gate can see that; it is why notebook 15 exists.")

### Gate 1 read back - white balance, from the pixels (L01)

Gate 1 of every protocol in this repo, and it is not skipped because the frames came from the
ASIAIR rather than from `asi.py`. `asi.neutralise_white_balance` runs when *this project* opens
the camera; the ASIAIR opens it independently, and a control this project never set is a control
this project cannot vouch for.

**The modal step between adjacent distinct values must be 16 on all four planes.** Greens at 16
with red at 17/18 and blue at 24 is the fingerprint of white balance still being applied, and
nothing captured after that fingerprint appears is usable.

Measured on the *stored* values, before any conversion - the check is what licenses the
conversion, so running it on converted data would be circular.

In [ ]:
probe_idx = np.linspace(0, len(kept) - 1, 8).astype(int)
gate1 = []
for i in probe_idx:
    r = kept.iloc[i]
    mosaic, _ = F.read(DATA / r.file)
    steps = {p: stats.value_step(v) for p, v in spatial.split(mosaic).items()}
    gate1.append({"file": r.file, "cell": r.cell, **steps})

gate1 = pd.DataFrame(gate1)
print(gate1.to_string(index=False))
bad = gate1[[p for p in PLANES]].ne(16).any(axis=1)
assert not bad.any(), (
    f"white balance is still being applied on {int(bad.sum())} of {len(gate1)} probed frames "
    "-- nothing in this session is usable (L01)")
print(f"\ngate 1 passed on {len(gate1)} frames spread across the night.")

## 2. `t_dead`, the constant this night was designed around

MISSION's model spends the sub-exposure question in two places: `R^2/t`, which says lengthen the
sub, and `t_dead`, which says the opposite. At fixed wall clock every sub costs its overhead
whether or not it collects anything, so `SNR ∝ sqrt(t/(t + t_dead))`.

It needs nothing but the headers, which is why it runs before anything that needs a constant.

**Published as a mean and a decomposition, never as a median** (rule 4). The archive's median gap
is 0.8 s and its mean is 19 s; a constant published as a median here would understate the overhead
by a factor of twenty.

**Interruptions are separated from overhead, and both are published.** A meridian flip and an
autofocus are not per-sub costs - they happen once and a few times - so averaging them into a
per-sub constant would inflate it for every sub in the model. The threshold that separates them is
fixed here before the numbers are read: **a gap more than three times the night's median gap is an
interruption**, and every one is listed.

**The decomposition into download and settle cannot be measured on this night, and the reason is
the protocol's own design.** The archive separated the two because it dithered every *second*
frame: the frames that skipped a dither showed the bare download, 0.68 s, flat across every gain
and exposure. This night dithered after every frame on purpose - so that no cell pays an overhead
the others do not - and the price of that choice is that **no frame here shows a download without
a settle on top of it.** What this night can prove is an *upper bound* on download: the shortest
gap observed. The split is published against the archive's independently measured 0.68 s, and the
bound beside it, with both named for what they are. The model consumes `t_dead`, which is measured
directly and needs no decomposition at all.

In [ ]:
seq = kept.sort_values("t").reset_index(drop=True)
gap = (seq.t.shift(-1) - seq.t).dt.total_seconds() - seq.exptime
seq["gap_s"] = gap
gaps = seq.dropna(subset=["gap_s"]).copy()

INTERRUPTION_FACTOR = 3.0
median_gap = float(gaps.gap_s.median())
cut = INTERRUPTION_FACTOR * median_gap
gaps["interruption"] = gaps.gap_s > cut

steady = gaps[~gaps.interruption]
t_dead = float(steady.gap_s.mean())

# Every frame dithered, so no gap here is a bare download.  The shortest one is
# an upper bound on download and nothing better is available from this night;
# the archive's 0.68 s is the independent figure, quoted as one.
download_bound = float(steady.gap_s.min())
ARCHIVE_DOWNLOAD_S = 0.68               # measured on 2002 archive frames, not ours -- L32's era
settle = t_dead - ARCHIVE_DOWNLOAD_S

print(f"median gap {median_gap:.2f} s; interruption cut at {cut:.2f} s")
print(f"{int(gaps.interruption.sum())} interruptions, "
      f"{gaps[gaps.interruption].gap_s.sum() / 60:.1f} min in total")
if gaps.interruption.any():
    print(gaps[gaps.interruption][["file", "cell", "minutes", "gap_s"]].round(1)
          .to_string(index=False))
print()
print(f"t_dead (steady state, mean)        {t_dead:7.2f} s   <- what the model consumes")
print(f"  shortest gap seen all night      {download_bound:7.2f} s   "
      "<- upper bound on download+save")
print(f"  archive's bare download          {ARCHIVE_DOWNLOAD_S:7.2f} s   "
      "<- independent, not ours")
print(f"  dither settle, by difference     {settle:7.2f} s")
print(f"t_dead including interruptions     {gaps.gap_s.mean():7.2f} s")
print("\nthe split is inferred, not measured: dithering after EVERY frame is what makes "
      "t_dead\none fair number, and it is also what stops this night showing a download "
      "without a settle.")
print()
print("per cell -- the interleave is only fair if these agree:")
by_cell = steady.groupby("cell").gap_s.agg(["count", "mean", "median", "std"])
print(by_cell.round(2).to_string())
cell_spread = float(by_cell["mean"].max() - by_cell["mean"].min())
print(f"\nspread across cells: {cell_spread:.2f} s "
      f"({100 * cell_spread / t_dead:.0f}% of t_dead)")
print("  " + ("small against the within-cell scatter; one t_dead is fair for all four"
              if cell_spread < steady.gap_s.std() else
              "COMPARABLE TO THE WITHIN-CELL SCATTER.  A cell that systematically waits longer "
              "pays an overhead the others do not, and t_dead_per_cell is the honest form"))

for t in (30.0, 120.0):
    print(f"\nat {t:.0f} s subs, t_dead = {t_dead:.1f} s means "
          f"{100 * t_dead / (t + t_dead):.0f}% of the night collects no photons")

## 3. The two ROIs

The signal ROI is fixed by the protocol and by four earlier sessions: 1024 x 1024 at (1408, 568).
Every SNR number comes from there.

The sky ROI is **chosen here, once, and then never moved.** The protocol says 512 x 512 in the
darkest corner, on no visible nebula, and the choice has to come from the data rather than from an
eye. The rule, fixed before the numbers: **the candidate is one of the four corner boxes, inset
far enough to clear the frame edge, and the winner is the one with the lowest modal level on the
green planes**, judged on a sample of the longest, highest-gain frames because those have the most
signal to separate the corners with.

`F_sky` from this field is an **upper bound** on true sky whichever corner wins, because
unresolved nebulosity is inside it and cannot be separated. It is published as such - and it is
the *right* bound for the model, which needs the level sitting under the faint signal, not the
zodiacal sky in the abstract.

In [ ]:
INSET = 64                               # clear the frame edge, amp glow and any overscan artefact
H, W = 2160, 3840


def corner_boxes(box=SKY_BOX, inset=INSET):
    """The four corners, even in origin and extent or the Bayer phase shifts (L05)."""
    xs, ys = (inset, W - box - inset), (inset, H - box - inset)
    out = {}
    for ny, y in zip(("top", "bottom"), ys):
        for nx, x in zip(("left", "right"), xs):
            out[f"{ny}-{nx}"] = (x & ~1, y & ~1, box, box)
    return out


def crop(a, roi):
    x, y, w, h = roi
    return a[y:y + h, x:x + w]


probe = (kept[(kept.gain == 200) & (kept.exptime == 120.0)]
         .sort_values("minutes").iloc[::6])
boxes = corner_boxes()
scout = []
for r in probe.itertuples():
    adc = stats.to_adc(F.read(DATA / r.file)[0]).astype(np.float64)
    for name, roi in boxes.items():
        planes = spatial.split(crop(adc, roi))
        scout.append({"file": r.file, "corner": name,
                      "green_mode": float(np.mean([stats.sky_level(planes[p])
                                                   for p in ("G1", "G2")]))})

scout = pd.DataFrame(scout)

# Compare corners WITHIN each frame, not across the night.  The sky level moved
# by tens of counts over five hours, which is an order of magnitude more than
# the corners differ from each other -- so a raw mean per corner buries the
# spatial question under the temporal one and ranks on noise.
scout["rel"] = scout.green_mode - scout.groupby("file").green_mode.transform("mean")
ranked = (scout.groupby("corner")
          .agg(raw_mean=("green_mode", "mean"), rel=("rel", "mean"),
               rel_sd=("rel", "std"), n=("rel", "count"))
          .sort_values("rel"))
print(f"probed {len(probe)} gain-200 120 s frames across the night.")
print("`rel` is the corner against the frame's own four-corner mean, which is what "
      "isolates\nthe spatial difference from the night's own drift:")
print(ranked.round(3).to_string())

SKY_CORNER = str(ranked.index[0])
SKY_ROI = boxes[SKY_CORNER]
corner_range = float(ranked.rel.max() - ranked.rel.min())
margin = float(ranked.rel.iloc[1] - ranked.rel.iloc[0])
# The uncertainty on a corner's mean, so the margin can be read against something.
se = float(ranked.rel_sd.iloc[:2].max() / np.sqrt(ranked.n.iloc[0]))
decisive = margin > se

print(f"\nsky ROI: {SKY_CORNER} corner at {SKY_ROI}")
print(f"  {margin:.2f} counts below the next corner, against a standard error of {se:.2f}")
print(f"  {corner_range:.2f} counts from darkest to brightest")
print(f"signal ROI: {SIGNAL_ROI}, unchanged from sessions 01, 02, 03 and 05")
print()
if not decisive:
    print("THE TWO DARKEST CORNERS ARE NOT SEPARABLE, so which of them wins is arbitrary and")
    print("the ROI is pinned by the rule rather than by the data.  That is fine -- the rule was")
    print("fixed in advance and either box measures the same sky -- but it is worth saying")
    print("rather than letting a ranked table imply a decision that was not made.")
if corner_range < 2.0:
    print("\nAll four corners are alike, and that is a finding about the FRAMING.  Gate 4 of")
    print("the protocol wanted the Gulf of Mexico -- genuinely dark structure -- in one corner,")
    print("so a sky ROI could sit on no nebula while the signal ROI sat on plenty.  Corners this")
    print("alike say the field was not framed that way, or that nebulosity reaches all four.")
else:
    print(f"\nThe brightest corner sits {corner_range:.1f} counts above the darkest, which is")
    print("real structure across the field.  F_sky below is measured in the darkest of them and")
    print("is an UPPER BOUND on true sky wherever it is taken: unresolved nebulosity is inside")
    print("the box and cannot be separated.  That is the right bound for the model, which wants")
    print("the level sitting under the faint signal, not the zodiacal sky in the abstract.")

## 4. Every frame, measured

One pass over the frames. For each, both ROIs, split RGGB, in ADC counts:

- the **modal level** of the sky ROI, per plane - what `F_sky` is built from (rule 3). Not the
  mean, which stars drag; not the median, which unresolved nebulosity drags.
- the **mean and spread** of the signal ROI, per plane - what the SNR numbers will be built from
  once there is an engine to stack with.
- the **clipped fraction** per plane, in both boxes, which is the star-colour constraint's raw
  material.

This is the expensive cell: it reads every frame once.

In [ ]:
def measure(path):
    adc = stats.to_adc(F.read(path)[0]).astype(np.float64)
    out = {}
    for tag, roi in (("sky", SKY_ROI), ("sig", SIGNAL_ROI)):
        for name, plane in spatial.split(crop(adc, roi)).items():
            out[f"{tag}_mode_{name}"] = (stats.sky_level(plane) if tag == "sky"
                                         else float(np.nan))
            out[f"{tag}_mean_{name}"] = float(plane.mean())
            out[f"{tag}_med_{name}"] = float(np.median(plane))
            out[f"{tag}_clip_{name}"] = float((plane >= FULL_SCALE).mean())
            out[f"{tag}_p999_{name}"] = float(np.percentile(plane, 99.9))
    return out


meas = []
for n, r in enumerate(kept.itertuples(), 1):
    meas.append({"file": r.file, **measure(DATA / r.file)})
    if n % 20 == 0:
        print(f"  {n}/{len(kept)}", flush=True)

frames = kept.merge(pd.DataFrame(meas), on="file")
frames["sky_mode_green"] = frames[["sky_mode_G1", "sky_mode_G2"]].mean(axis=1)
print(f"\nmeasured {len(frames)} frames")
print(frames.groupby("cell")[["sky_mode_R", "sky_mode_green", "sky_mode_B"]]
      .mean().round(3).to_string())

### The rejection rule, fixed before the numbers were seen

Gate 3 of the protocol wants guide RMS per frame and a threshold written down in advance. **The
ASIAIR does not write guide RMS into the FITS header**, and there is no per-frame guiding record
in `data/session06`, so that half of the gate cannot be run at all. It is published as not
measured rather than replaced by something that sounds like it.

What *can* be tested for is cloud, which is the failure the gate mostly exists to catch: cloud
scatters town light and lifts the sky level, or occults the field and drops it. The rule, and it
is applied identically to all four cells:

**A frame is rejected when its sky mode departs from its own cell's median by more than four
times that cell's own MAD.** Per cell, because the cells sit at different levels by construction;
against a MAD, because the thing being detected is an outlier and the estimator must not be
dragged by it.

The count rejected per cell is published. **If the four cells lose materially different numbers,
the interleave has been broken** and that is reported rather than averaged over.

In [ ]:
REJECT_SIGMAS = 4.0

frames["sky_dev"] = np.nan
for cell, d in frames.groupby("cell"):
    med = d.sky_mode_green.median()
    mad = stats.MAD_TO_SIGMA * np.median(np.abs(d.sky_mode_green - med))
    frames.loc[d.index, "sky_dev"] = (d.sky_mode_green - med) / max(mad, 1e-9)

frames["rejected"] = frames.sky_dev.abs() > REJECT_SIGMAS
frames["reject_reason"] = np.where(frames.rejected, "sky level outlier", "")

rej = frames.groupby("cell").agg(frames_n=("file", "count"), rejected=("rejected", "sum"))
rej["kept"] = rej.frames_n - rej.rejected
rej["rejected_pct"] = 100 * rej.rejected / rej.frames_n
print(rej.round(2).to_string())

spread = float(rej.rejected_pct.max() - rej.rejected_pct.min())
print(f"\nspread in rejection rate across cells: {spread:.1f} points")
print("  " + ("the four cells lose comparable numbers, so the interleave survives rejection"
              if spread < 10 else
              "THE CELLS LOSE DIFFERENT NUMBERS.  The interleave is broken by the rejection "
              "rule, and every ratio below inherits that -- report it, do not average it"))
print(f"\nguide RMS: not in the headers, so gate 3's guiding half is unmeasured (see above)")
good = frames[~frames.rejected].copy()

## 5. `F_sky`, in electrons, per plane

Rule 3, and the reason the whole night exists. The sky rate is

```
F_sky = (sky mode - pedestal) * g(gain) / t        e-/px/s
```

**The constants come from `cold_constants.json` and nowhere else.** Session 02's `g` and session
01's pedestal are at -10 C and this night is at -20 C; substituting them would be exactly the
silent substitution this repo forbids, and notebook `15` exists to remove it. This cell refuses to
run without that file rather than quietly reaching for the warm one.

**What the missing bias brackets cost, stated plainly.** The protocol asks for 20 bias frames at
each gain at each end of the night, so the pedestal comes from *this* night and the offset state
can be assigned per gain. There are none. The pedestal used is notebook `15`'s bench block at the
same setpoint, same gain and same offset - the right temperature, on the wrong night. Session 03
found the pedestal hops by about one count on 4% of frames, and session 04 traced it to something
exposure-correlated, so the error this leaves is of that order: **about a count**, which at gain
50 and 30 s subs is a large fraction of the sky signal itself. It is carried into the published
uncertainty rather than ignored.

In [ ]:
COLD = RESULTS / "cold_constants.json"
assert COLD.exists(), (
    f"{COLD} does not exist.  This night was shot at {SETPOINT_C} C and every published g and "
    "pedestal in this repo is at -10 C.  Run protocols/07-cold-constants.md and "
    "notebooks/15_cold_constants.ipynb first -- do not substitute the warm constants "
    "(CLAUDE.md: never substitute a mismatched dark, flat or constant silently)")

K7 = json.loads(COLD.read_text())
assert float(K7["setpoint"]["value"]) == SETPOINT_C, (
    f"cold_constants.json is at {K7['setpoint']['value']} C, this night at {SETPOINT_C} C")

G = {int(k): v for k, v in K7["system_gain_cold"]["value"].items()}
G_PLANE = {int(k): v for k, v in K7["system_gain_cold_per_plane"]["value"].items()}
PEDESTAL = {int(k): v for k, v in K7["pedestal_cold"]["value"].items()}
R_E = {int(k): v for k, v in K7["read_noise_cold_e"]["value"].items()}
PEDESTAL_UNCERTAINTY = 1.0        # counts; the offset state's own step, session 03

missing = [g for g in good.gain.unique() if g not in G or g not in PEDESTAL]
assert not missing, f"cold_constants.json has no g or pedestal for gains {missing}"

# Computed on every frame, rejected ones included, so that sky_frames.csv is a
# record of the night rather than of the subset that survived a rule.  `good` is
# re-derived from it afterwards and is what every published number uses.
for p in PLANES:
    g_of = frames.gain.map(lambda x: G_PLANE[x][p])
    frames[f"F_sky_{p}"] = ((frames[f"sky_mode_{p}"] - frames.gain.map(PEDESTAL))
                            * g_of / frames.exptime)
    frames[f"F_sky_err_{p}"] = PEDESTAL_UNCERTAINTY * g_of / frames.exptime

frames["F_sky_green"] = frames[["F_sky_G1", "F_sky_G2"]].mean(axis=1)
good = frames[~frames.rejected].copy()

sky = good.groupby("cell").agg(**{
    f"{p}": (f"F_sky_{p}", "mean") for p in PLANES})
sky["green"] = good.groupby("cell").F_sky_green.mean()
sky["pedestal_err_e_s"] = good.groupby("cell")[[f"F_sky_err_{p}" for p in PLANES]].mean().mean(axis=1)
print("F_sky per cell, e-/px/s:")
print(sky.round(4).to_string())

print("\nper plane, over the whole night (the number the model consumes):")
overall = pd.Series({p: good[f"F_sky_{p}"].mean() for p in PLANES})
overall["green"] = good.F_sky_green.mean()
print(overall.round(4).to_string())

### L32, reproduced or refuted

The one `LEGACY` entry this session consumes. The retired project claimed **1.594 e-/px/s green**
(R 1.500, B 0.910) at f/4.8, 2.27"/px, unfiltered, near zenith, suburban Bortle 5-6 - implying
about 19.1 mag/arcsec^2 - and was explicit that it was "a rate for that night, at that altitude",
falling about 5% across two hours as the target rose.

So there are two claims, and reproducing the *variation* is as much the test as reproducing the
number. Both are checked below.

The comparison is not apples to apples in one respect worth naming: L32's figure came from a
different night under the same sky, and sky brightness is weather, moon and town lighting. A
disagreement of tens of percent is a different sky, not a wrong measurement. What would refute it
is a disagreement in *shape* - a rate that climbs while the target climbs, say.

In [ ]:
L32 = {"R": 1.500, "green": 1.594, "B": 0.910}

cmp = pd.DataFrame({"measured": overall, "L32": pd.Series(L32)}).dropna()
cmp["ratio"] = cmp.measured / cmp.L32
cmp["pct"] = 100 * (cmp.ratio - 1)
print(cmp.round(4).to_string())

half = good.minutes.median()
early, late = good[good.minutes < half], good[good.minutes >= half]
drift_pct = 100 * (late.F_sky_green.mean() / early.F_sky_green.mean() - 1)
fit = np.polyfit(good.minutes, good.F_sky_green, 1)
print(f"\nvariation across the night: {drift_pct:+.1f}% from first half to second")
print(f"  trend {fit[0] * 60:+.4f} e-/px/s per hour on green, over "
      f"{good.minutes.max() / 60:.1f} h")
print(f"  L32 reported about -5% across two hours as its target rose")

print("\nMISSION's third assumption -- that the dimmest and brightest planes differ:")
planes_only = overall[PLANES]
gapped = 100 * (planes_only.max() / planes_only.min() - 1)
print(f"  brightest {planes_only.idxmax()} {planes_only.max():.4f}, "
      f"dimmest {planes_only.idxmin()} {planes_only.min():.4f}  ({gapped:.1f}% apart)")
print("  " + ("HOLDS -- the exposure floor and the clipping ceiling are set by different "
              "planes, and the per-plane Pareto framing buys something"
              if gapped > 10 else
              "DOES NOT HOLD at this separation.  If R, G and B agree, the per-plane framing "
              "buys nothing and MISSION's third assumption needs revisiting"))

## 6. Star clipping, counted rather than eyeballed

Rule 7, in the form this repo can currently support.

The protocol asks for the fraction of *stars* with a clipped core, per plane, per cell, against
**the same detection list**. That is not reachable yet and the reason is worth being precise
about: the same physical star sits on a different pixel in every frame, because the night dithered
after every one. Matching a detection list across frames is registration, and registration is
build step 5.

What is reachable, and is a real measurement rather than a placeholder, is the **pixel-level**
count: what fraction of each plane has reached the top code, and how close the bright tail gets.
It answers the Pareto question directionally - gain 200 has a sixth of the well, so it must clip
far more of the same field at matched sub length - and it does so without a detector.

It understates the star-level number, and in a knowable direction: a star whose core clips
occupies a handful of pixels, so a clipped *fraction* of 0.1% is a great many clipped stars.

In [ ]:
clip = good.groupby("cell").agg(**{
    **{f"clip_{p}": (f"sig_clip_{p}", "mean") for p in PLANES},
    **{f"p999_{p}": (f"sig_p999_{p}", "mean") for p in PLANES},
})
clip["headroom_pct"] = 100 * clip[[f"p999_{p}" for p in PLANES]].max(axis=1) / FULL_SCALE
print("signal ROI, mean over frames in each cell:")
print(clip.round(5).to_string())

print("\nthe Pareto point, at matched sub length:")
for t in (30.0, 120.0):
    a = clip.loc[CELL_NAME[(50, t)], [f"clip_{p}" for p in PLANES]].mean()
    c = clip.loc[CELL_NAME[(200, t)], [f"clip_{p}" for p in PLANES]].mean()
    ratio = c / a if a > 0 else np.inf
    print(f"  {t:.0f} s: gain 50 clips {a:.5f} of pixels, gain 200 clips {c:.5f} "
          f"({ratio:.1f}x)")
print("\nper plane at 120 s -- the brightest plane sets the ceiling (MISSION):")
print(clip.loc[[CELL_NAME[(50, 120.0)], CELL_NAME[(200, 120.0)]],
               [f"clip_{p}" for p in PLANES]].round(6).to_string())

## 7. The three things that need an integration engine

`eta_comb` on registered lights, the SNR estimator's repeatability, and the four ranked pairs all
require the frames to be **registered and integrated**. This repo has no engine for that:
`astropix/pixinsight.py` does not exist and neither does `pjsr/`. That is build step 5, and
`LEGACY.md` still holds its six entries - L16 to L24 - waiting for it.

They are published as **nulls with reasons**, which in this repo is a result rather than a gap.
What replaces them meanwhile is the part that *can* be computed from measured numbers: each pair's
**predicted** separation, using this session's own `F_sky` and `t_dead` instead of the inherited
ones the protocol was written with.

That substitution matters more than it sounds. The protocol's table bracketed `t_dead` between
0.7 s and 19 s and noted that the C -> D pair is a near-tie at 1.6% if dead time is small and a
20% separation if it is not. This night measured `t_dead` directly, and section 2 has the answer -
so the question of **which pairs are still tests at all** is now settled by measurement rather
than by a bracket.

MISSION is explicit that a pair the model calls a tie is a null result every model passes. Until
the repeatability of the SNR estimator is measured, no separation below can be called a test
either - the bar is unknown. Both facts are carried in the published table.

In [ ]:
D_BOUND = 0.000534                  # e-/px/s at -10 C, session 03; colder here, so an upper bound


def snr_rel(t, F_sky, R_e, t_dead, D=D_BOUND):
    """MISSION's model, up to the constants that cancel in a ratio.

    SNR(T_night, t) ∝ sqrt(t / (t + t_dead)) / sqrt(F_sky + D + R^2/t)

    F_obj cancels only for the *same* target, which is what a pair is, and
    eta_comb cancels only if it is the same for both settings -- which is an
    assumption, and one this session could not test.  It is named in the
    published note rather than buried here.
    """
    return np.sqrt(t / (t + t_dead)) / np.sqrt(F_sky + D + R_e ** 2 / t)


PAIRS = [("A", "B", "t-pair at gain 50"), ("C", "D", "t-pair at gain 200"),
         ("A", "C", "straddles HCG, t = 30 s"), ("B", "D", "straddles HCG, t = 120 s")]
CELL_OF = {v: k for k, v in CELL_NAME.items()}

rows = []
for first, second, why in PAIRS:
    for plane in ("green", "B"):
        col = "F_sky_green" if plane == "green" else "F_sky_B"
        out = {}
        for tag in (first, second):
            g, t = CELL_OF[tag]
            out[tag] = snr_rel(t, float(good[good.cell == tag][col].mean()), R_E[g], t_dead)
        rows.append({
            "pair": f"{first}->{second}", "why": why, "plane": plane,
            "t_dead_s": t_dead,
            "predicted_pct": 100 * (out[second] / out[first] - 1),
            "measured_pct": None,
            "repeatability_pct": None,
            "verdict": "pending registration",
        })

pairs = pd.DataFrame(rows)
print(pairs.round(3).to_string(index=False))
print(f"\nall four predictions use the measured t_dead of {t_dead:.1f} s and this session's own")
print("F_sky, not L32's.  The protocol's own table bracketed t_dead at 0.7 s and 19 s.")
print("\nnothing in the measured column can be filled without build step 5:")
print("  eta_comb on registered lights  -- needs registration and integration")
print("  SNR estimator repeatability    -- needs two half-stacks per cell")
print("  the four ranked pairs          -- needs a stacked SNR to rank")

### The dark ladder this session was meant to sit beside

Session 03 published `eta_comb` on **darks**: no registration, no resampling, so it is an upper
bound by construction. The gap between that and the same measurement on registered lights **is**
the resampling loss, and it is the number this session was supposed to add.

With no engine there is no gap to measure. The dark ladder is printed here so the missing column
is visible rather than merely absent, and so the next session knows exactly what shape the answer
takes.

In [ ]:
K3 = json.loads((RESULTS / "dark_constants.json").read_text())
dark_ladder = {int(k): v for k, v in K3["eta_comb"]["value"].items()}
LADDER_N = [2, 4, 8, 16, 32]

print(f"{'N':>4} {'darks (session 03)':>20} {'registered lights':>20} {'resampling loss':>18}")
for n in LADDER_N:
    print(f"{n:4d} {dark_ladder.get(n, float('nan')):20.4f} {'not measured':>20} "
          f"{'not measured':>18}")
print(f"\nthe ladder needs {max(LADDER_N)} frames per cell; this night has "
      f"{int(per_cell.min())} at worst, so the data is there and only the engine is missing.")

## 8. Publishing

`sky_constants.json`, `sky_frames.csv` and `sky_pairs.csv`, with provenance on every constant and
a stated reason on every null.

**L32 does not leave `LEGACY.md` from this cell.** The entry says it lands in `results/` "as the
working suburban sky rate, with its variability stated", and both halves are published below - so
harvesting it is correct and it is done in the same commit as this notebook, by hand, since
`LEGACY.md` is prose and not a file a notebook may write.

In [ ]:
measured_on = str(good.date_obs.min())[:10]
n_frames = len(good)


def constant(value, unit, uncertainty, note):
    return {"value": value, "unit": unit, "uncertainty": uncertainty,
            "source_frames": n_frames, "measured_on": measured_on,
            "notebook": "17_sky_pair.ipynb", "note": note}


def nn(v):
    return None if v is None or not np.isfinite(v) else round(float(v), 6)


NEEDS_STEP5 = ("not measured.  This quantity requires the frames to be registered and "
               "integrated, and this repo has no engine for that: astropix/pixinsight.py "
               "does not exist and neither does pjsr/ (build step 5, LEGACY L16-L24).  The "
               "frames are on disk and sufficient -- only the engine is missing")

constants = {
    "F_sky": constant(
        {p: nn(overall[p]) for p in PLANES}, "e-/px/s",
        {p: nn(good[f"F_sky_err_{p}"].mean()) for p in PLANES},
        "rule 3 of protocols/06-sky-pair.md: the modal level of the sky ROI, pedestal "
        f"subtracted, x g(gain), per frame, averaged over the night.  Sky ROI is the "
        f"{SKY_CORNER} corner, {SKY_BOX}x{SKY_BOX} at {SKY_ROI}, chosen once as the darkest of "
        "the four corners and then never moved.  An UPPER BOUND on true sky: unresolved "
        "nebulosity is inside the box and cannot be separated -- and it is the right bound for "
        "the model, which needs the level sitting under the faint signal.  The uncertainty is "
        "the pedestal's alone, at one count, because no bias bracket was shot on the night and "
        "the pedestal comes from notebook 15's bench block at the same setpoint and gain"),
    "F_sky_green": constant(
        nn(good.F_sky_green.mean()), "e-/px/s", nn(good.F_sky_green.std()),
        "the mean of G1 and G2, which is the number L32 is quoted in and the one a single-figure "
        "sky rate means.  The uncertainty here is the frame-to-frame scatter across the night, "
        "not the pedestal error -- it is the variability, which is the thing L32 says matters"),
    "F_sky_per_cell": constant(
        {c: {p: nn(v) for p, v in sky.loc[c, PLANES].items()} for c in sky.index},
        "e-/px/s", None,
        "the same rate measured independently in each of the four cells.  They share a sky, so "
        "this is a consistency check on the estimator across two gains and two sub lengths "
        "rather than four measurements of four things"),
    "F_sky_trend": constant(
        nn(fit[0] * 60), "e-/px/s per hour, green", None,
        f"how the sky rate moved across {good.minutes.max() / 60:.1f} h as the target transited: "
        f"{drift_pct:+.1f}% from the first half of the night to the second.  L32 reported about "
        "-5% across two hours as its target rose, and reproducing that shape is as much the test "
        "as reproducing the number"),
    "L32_comparison": constant(
        {k: {"measured": nn(cmp.measured[k]), "L32": nn(cmp.L32[k]),
             "pct": nn(cmp.pct[k])} for k in cmp.index},
        "e-/px/s and % against L32", None,
        "LEGACY L32, reproduced or refuted on our own frames.  Sky brightness is weather, moon "
        "and town lighting, so a disagreement of tens of percent is a different night and not a "
        "wrong measurement; what would refute the claim is a disagreement in shape.  L32's "
        "figure came from a codebase with no provenance discipline and cannot be re-read at its "
        "origin, so this is the only check it will ever get"),
    "planes_differ": constant(
        nn(gapped), "% between the brightest and dimmest CFA plane", None,
        "MISSION's third assumption: the exposure floor is set by the dimmest plane and the "
        "clipping ceiling by the brightest, so if R, G and B agree the per-plane Pareto framing "
        "buys nothing.  Measured on F_sky above"),
    "t_dead": constant(
        nn(t_dead), "s", nn(steady.gap_s.std()),
        f"rule 4: the mean frame-to-frame DATE-OBS gap minus exposure, in steady state, over "
        f"{len(steady)} intervals.  A MEAN and never a median -- the median is "
        f"{median_gap:.2f} s here and the model consumes the mean, so a median would understate "
        f"the overhead.  Measured under this night's dither cadence, which was every frame; the "
        "archive dithered every second frame and measured 17-20 s"),
    "t_dead_decomposition": constant(
        {"download_and_save_bound": nn(download_bound),
         "download_and_save_archive": ARCHIVE_DOWNLOAD_S,
         "dither_settle_inferred": nn(settle)},
        "s", None,
        "rule 4's decomposition, and it is INFERRED rather than measured -- read the three "
        "numbers as what they each are.  This night dithered after every frame, deliberately, so "
        "that no cell pays an overhead the others do not; the price is that no gap here shows a "
        "download without a settle on top of it.  download_and_save_bound is the shortest gap "
        "seen all night, which is the only upper bound this night can prove. "
        "download_and_save_archive is the 0.68 s the archive measured, flat across every gain and "
        "exposure, on the frames that skipped a dither -- an independent number from the historic "
        "corpus and not ours.  dither_settle_inferred is t_dead minus that.  The model consumes "
        "t_dead, which is measured directly and needs none of this"),
    "t_dead_including_interruptions": constant(
        nn(gaps.gap_s.mean()), "s", None,
        f"the same mean with the {int(gaps.interruption.sum())} interruptions left in -- the "
        f"meridian flip and any autofocus.  Separated because they are not per-sub costs: they "
        f"happen once or a few times, and averaging them into a per-sub constant would inflate "
        f"it for every sub the model ever considers.  The cut is a gap over "
        f"{INTERRUPTION_FACTOR}x the night's median, fixed before the numbers were read"),
    "t_dead_per_cell": constant(
        {c: nn(v) for c, v in by_cell["mean"].items()}, "s", nn(cell_spread),
        f"the interleave is only fair if these agree: a cell that systematically sits after a "
        f"dither pays an overhead the others do not, and that overhead lands inside t_dead where "
        f"it cannot be separated from the setting.  The spread across the four is "
        f"{cell_spread:.2f} s, against a within-cell scatter of {steady.gap_s.std():.2f} s -- "
        f"read the two together before using the single t_dead for a cell-to-cell ratio"),
    "dither_cadence": constant(
        "every frame", "frames per dither", None,
        "the one place this protocol departs from the archive, and the reason is a confound "
        "rather than a preference.  eta_comb's registration half is measured against the number "
        "of distinct dither positions, so this is part of that constant's provenance"),
    "eta_comb_registered": constant(None, "measured sd reduction against ideal sqrt(N)", None,
                                    "rule 5.  " + NEEDS_STEP5 + ".  Session 03's dark ladder is "
                                    "in dark_constants.json and is an upper bound by "
                                    "construction -- no registration, no resampling -- so the "
                                    "gap between the two IS the resampling loss, and that gap is "
                                    "what this session was meant to add"),
    "snr_repeatability": constant(None, "% difference between two half-stacks", None,
                                  "rule 6.  " + NEEDS_STEP5 + ".  Until it exists no predicted "
                                  "separation can be called a test: MISSION requires the "
                                  "separation to exceed the estimator's own repeatability, and "
                                  "that bar is currently unknown"),
    "ranked_pairs": constant(None, "the ranking verdict on four pairs", None,
                             "MISSION's definition of done.  " + NEEDS_STEP5 + ".  The predicted "
                             "separations from this session's own measured constants are in "
                             "sky_pairs.csv, with the measured column empty"),
    "guide_rms": constant(None, "arcsec", None,
                          "gate 3 of the protocol wants guide RMS logged per frame and a "
                          "rejection threshold fixed in advance.  The ASIAIR does not write "
                          "guide RMS into the FITS header and no separate per-frame guiding log "
                          "was kept, so this half of the gate could not be run.  What was run "
                          "instead is the cloud test in sky_frames.csv, on the sky level itself"),
    "rejection": constant(
        {c: {"frames": int(rej.frames_n[c]), "rejected": int(rej.rejected[c])}
         for c in rej.index},
        "frames per cell", None,
        f"a frame is rejected when its sky mode departs from its own cell's median by more than "
        f"{REJECT_SIGMAS} times that cell's MAD -- a cloud test, applied identically to all four "
        f"cells, with the rule fixed before the numbers were seen.  The spread in rejection rate "
        f"across cells is {spread:.1f} points; materially different losses would mean the "
        "interleave had been broken by the rule itself"),
    "frames_dropped_to_flip": constant(
        int(night.flipped.sum()), "frames", None,
        f"the field rotated from {sorted(night.rotator.unique())} degrees about "
        f"{float(night[night.flipped].minutes.max()):.0f} minutes in -- a meridian flip.  Gate 4 "
        "fixes both ROIs for the night and a flip moves them, so the pre-flip frames are dropped "
        "rather than given a second pair of ROIs.  Every cell still clears the protocol's floor "
        "of 24"),
    "setpoint_deviation": constant(
        {"protocol_c": -10.0, "actual_c": SETPOINT_C,
         "frames_outside_band": int(kept.temp_flag.sum())},
        "C", None,
        "the night ran at a -20 C setpoint against the protocol's -10 C, on every frame.  No "
        "per-frame gate can see that -- the cooler held its own setpoint perfectly.  It is why "
        "every constant above is computed from cold_constants.json and not from "
        "ptc_constants.json, and why notebook 15 exists at all"),
    "clipped_fraction": constant(
        {c: {p: nn(clip.loc[c, f"clip_{p}"]) for p in PLANES} for c in clip.index},
        "fraction of pixels at the top code, signal ROI", None,
        "rule 7, in the form this repo can currently support.  The rule asks for the fraction of "
        "STARS with a clipped core against the same detection list; the same physical star sits "
        "on a different pixel in every frame because the night dithered after each one, so "
        "matching a list across frames is registration and registration is build step 5.  This "
        "is the pixel-level count instead: a real measurement, and one that understates the "
        "star-level number in a knowable direction, since a clipped core occupies only a handful "
        "of pixels"),
    "sky_roi": constant(
        {"corner": SKY_CORNER, "roi": list(SKY_ROI), "signal_roi": list(SIGNAL_ROI),
         "corner_range_counts": nn(corner_range),
         "margin_over_second_counts": nn(margin),
         "margin_standard_error_counts": nn(se)},
        "x, y, w, h in mosaic pixels", None,
        f"gate 4.  The signal ROI is the box sessions 01, 02, 03 and 05 used, so a number from it "
        f"is comparable with the bench.  The sky ROI was chosen once, as the darkest of the four "
        f"inset corner boxes judged on the green modal level relative to each frame's own "
        f"four-corner mean -- relative, because the sky moved by tens of counts across the night "
        f"and a raw mean would rank on that instead -- and then never moved.  The margin over the "
        f"runner-up is {margin:.2f} counts against a standard error of {se:.2f}, so "
        + ("the two darkest corners are NOT separable and which of them won is arbitrary: the "
           "rule picked the box, not the data.  Either measures the same sky"
           if margin <= se else
           "the choice is resolved by the data") +
        f".  Darkest to brightest is {corner_range:.2f} counts"),
}

with open(CONSTANTS, "w", encoding="utf8") as fh:
    json.dump(constants, fh, indent=2)

keep_cols = [c for c in frames.columns if not c.startswith(("sig_mode",))]
frames[keep_cols].to_csv(FRAMES_CSV, index=False)
pairs.to_csv(PAIRS_CSV, index=False)

print(f"wrote {CONSTANTS} with {len(constants)} constants")
nulls = [k for k, v in constants.items() if v["value"] is None]
print(f"  {len(constants) - len(nulls)} measured, {len(nulls)} published null with a reason: "
      + ", ".join(nulls))
print(f"wrote {FRAMES_CSV}  ({len(frames)} rows)")
print(f"wrote {PAIRS_CSV}  ({len(pairs)} rows, measured column empty pending build step 5)")
print()
print(f"{n_frames} frames after the flip drop and the cloud rejection, captured {measured_on}")
print("L32 is harvested by hand in the same commit: LEGACY.md is prose, not a file a "
      "notebook may write.")